In [2]:
import pyaudio        # Library to capture audio from microphone
import wave           # to store, read, and manage raw audio data in WAV format, which is ideal for further processing (librosa, MFCC, CNN models)

#to record and store the wav
def microphone_audio_input(output_file=r"C:\Users\pavani\Desktop\python\output.wav", duration=5, s=16000, channels=1):
    CHUNK = 1024  # Buffer size - Number of audio samples per frame
    FORMAT = pyaudio.paInt16  # 16-bit resolution
    RATE = s   # 16 kHz sampling rate

    audio = pyaudio.PyAudio()   # Creates PyAudio object
    
    # Open microphone stream
    stream = audio.open(format=FORMAT, 
                        channels=channels, 
                        rate=RATE,
                        input=True,   # Enable microphone input
                        frames_per_buffer=CHUNK)   # Buffer size

    print(f"Recording started for {duration} seconds...")      # Indicate start of recording
    
    frames = []     # List to store recorded audio chunks

    # Read audio data from microphone for given duration
    for _ in range(int(RATE / CHUNK * duration)):
        data = stream.read(CHUNK) # Read one chunk 
        frames.append(data)     # store the chunk

    print("Recording finished.")
    
    stream.stop_stream()     # Stop audio stream
    stream.close()           # Close stream
    audio.terminate()        # Terminate PyAudio instance

    # Save the recorded audio as WAV and to write WAV file correctly
    with wave.open(output_file, 'wb') as wf:
        wf.setnchannels(channels)
        wf.setsampwidth(audio.get_sample_size(FORMAT))
        wf.setframerate(RATE)
        wf.writeframes(b''.join(frames))
    print(f"Saved recording to - {output_file}")

    return output_file

In [3]:
import warnings
warnings.filterwarnings("ignore")

In [4]:
# audio preprocessing

import librosa        # for audio processing utilities
import numpy as np          # for numerical operations and array handling
import librosa.effects      # for audio normalizing
import noisereduce as nr    # for reducing noise in the audio

def preprocess_audio(file_path, s = 16000, duration = 5):
    # Load audio (mono) - stereo adds redundant data, mono reduces data size and computation
    signal, _ = librosa.load(file_path, sr = s, mono = True)
    #sr is sample rate we want to load per second 
    #signal is a numpy array 1D array, it has sr*t = 16000*5 = 80,000 values. These values are the amplitude of waveform

    # Trim silence - silence carries no useful audio information
    signal, _ = librosa.effects.trim(y = signal, top_db = 35)      #choosing 35 which is somewhat in the middle
    #Lower top_db(20)-stricter silence removal(only trims very quiet parts)
    #Higher top_db(60)-more aggressive (trims even moderately quiet parts)

    # Noise reduction - background noise confuses models, hides audio patterns, noise varies across environments
    signal = nr.reduce_noise(y = signal, sr = s)    #gives only one value
    
    # Normalizing the volume - different recordings have different loudness, models should learn audio patterns, not volume differences
    signal = librosa.util.normalize(S = signal)

    # Pad or truncate - to make all audio samples the same length
    desired_length = s * duration
    if len(signal) < desired_length:
        signal = np.pad(signal, (0, desired_length - len(signal)))
    else:
        signal = signal[:desired_length]

    return signal

In [5]:
#Feature extraction

def extract_features(signal, s = 16000, n_mfcc=20):
    
    mfcc = librosa.feature.mfcc(y = signal, sr = s, n_mfcc=n_mfcc)
    
    centroid = librosa.feature.spectral_centroid(y=signal, sr=s)
    # centroid is high-the sound is bright or sharp - female
    # centroid is low-the sound is dull or bassy - male
    
    zcr = librosa.feature.zero_crossing_rate(y = signal)
    # High ZCR - Noisy, sharp, or high-pitched sounds - female
    # Low ZCR - Smooth, low, bassy sounds - male
     
    rolloff = librosa.feature.spectral_rolloff(y = signal, sr=s)
    # if spectral rolloff is low means - Male voice(Deep, Low-pitched)
    # if spectral rolloff is high means - Female voice(high-pitched)
    
    rmse = librosa.feature.rms(y = signal) 
    # helps in trimming silent segments
    
    features = np.vstack([mfcc, centroid, zcr, rolloff, rmse])    # combining all these extracted features
    
    return features  

In [6]:
#load the dataset

import os                 # To navigate through folders and file paths  

def load_voice_dataset(base_path="C:\\Users\\pavani\\Desktop\\python\\voices"):
    # X will store extracted features of each audio file
    X = []
    # y will store corresponding labels (0 = male, 1 = female)
    y = []

    # label mapping: folder name to numeric label for model training
    labels = {
        "male": 0,
        "female": 1
    }

    # looping through each class (male and female)
    for label_name, label_id in labels.items():
        
        # creates full path to the class folder (voice/male n voice/female)
        folder_path = os.path.join(base_path, label_name)

        # looping through every file in the folder
        for file in os.listdir(folder_path):
            
            # processes only WAV audio files
            if file.endswith(".wav"):
                
                # creates full path to the audio file
                file_path = os.path.join(folder_path, file)

                # preprocessing the - load audio, convert to mono, trim silence, reduce noise, normalize, pad or truncate to fixed length
                signal = preprocess_audio(file_path = file_path)
                # extracting the audio features(MFCC + spectral features)
                features = extract_features(signal = signal)

                # stores the extracted features
                X.append(features)
                # stores the corresponding label (male/female)
                y.append(label_id)

    # converting those feature list to numpy array
    X = np.array(X)
    # converting labels list to numpy array
    y = np.array(y)

    # returns the dataset (features, labels)
    return X, y

In [18]:
# # build CNN classifier architecture

# from tensorflow.keras.models import Sequential            # imports the sequential API to build a neural network layer-by-layer
# from tensorflow.keras.layers import Conv2D, MaxPooling2D, BatchNormalization
# from tensorflow.keras.layers import Flatten, Dense, Dropout
# from tensorflow.keras.optimizers import Adam

# def building_cnn_classifier(input_shape, num_of_classes = 2):
# 	model = Sequential()

# # input layer
# #2D convolution layer to extract spatial features
# 	model.add(Conv2D(filters = 32, 			# low level patterns - pitch edges
# 			kernel_size = (3,3),		# small window size
# 			activation = 'relu',            # adds non-linearity
# 			input_shape = input_shape))      # (features(MFCC+spectralFeatures, no of frames, channels) like grayscaling
# 	model.add(BatchNormalization())			# normalizes activations to speed up training, so that activation will be wellscaled
# 	model.add(MaxPooling2D(pool_size = (2,2)))		# reduces spatial dimensions, keeps important features
	
#     model.add(Conv2D(64, (3,3), activation = 'relu'))		# more filtering for complex patterns
# 	model.add(BatchNormalization())
# 	model.add(MaxPooling2D((2,2)))
# # full conneceted layer
# 	model.add(Flatten())					# converts 2D/3D feature maps into a 1D vector for dense layers
# 	model.add(Dense(32, activation = 'relu'))			# fully connected layer used for classification or decision making
# 	model.add(Dropout(0.2))					# randomly drops neurons during training to prevent overfitting
# # output layer
# 	model.add(Dense(num_of_classes, activation = 'softmax'))		# outputs probability of each class - m/f
# # model compilation
# 	model.compile(optimizer = Adam(learning_rate = 0.01), 		# adaptive learning-rate optimizer for faster and more stable convergence
# 			loss = 'categorical_crossentropy',		# for integer labels (0,1)
# 			metrics = ['accuracy'])				# keeps classification accuracy

# 	return model

In [9]:
# from sklearn.model_selection import train_test_split
# from tensorflow.keras.utils import to_categorical

# # load the dateset( features, labels)
# X, y = load_voice_dataset()
# # converting categorical variables into a binary format
# y = to_categorical(y, num_classes = 2)
# # reshaping the samples to CNN 4D input
# X = np.expand_dims(X, axis = -1)                      # - (samples, features, time frames, channel = 1(mono)) like grayscaling

# # train/test split
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.4, random_state = 40, shuffle = True)

# # input shape for cnn (features, frames, channels)
# input_shape = X_train.shape[1:]
# # building cnn model
# model = building_cnn_classifier(input_shape = input_shape)

# # training the model
# model.fit(
#     X_train,                # training features
#     y_train,                # training labels
#     validation_split=0.2,   # 20% of training data used for validation
#     epochs=10,              # number of training iterations
#     batch_size=32)         # samples processed per update
# # evaluating the model
# loss, accuracy = model.evaluate(X_test, y_test)
# print("Test Accuracy:", accuracy)

Epoch 1/10
243/243 ━━━━━━━━━━━━━━━━━━━━ 20s 74ms/step - accuracy: 0.8592 - loss: 0.3436 - val_accuracy: 0.9401 - val_loss: 0.2136
Epoch 2/10
243/243 ━━━━━━━━━━━━━━━━━━━━ 19s 76ms/step - accuracy: 0.9745 - loss: 0.0793 - val_accuracy: 0.8947 - val_loss: 0.2768
Epoch 3/10
243/243 ━━━━━━━━━━━━━━━━━━━━ 18s 73ms/step - accuracy: 0.9790 - loss: 0.0576 - val_accuracy: 0.8762 - val_loss: 0.3740
Epoch 4/10
243/243 ━━━━━━━━━━━━━━━━━━━━ 18s 75ms/step - accuracy: 0.9772 - loss: 0.0639 - val_accuracy: 0.9097 - val_loss: 0.2647
Epoch 5/10
243/243 ━━━━━━━━━━━━━━━━━━━━ 19s 78ms/step - accuracy: 0.9835 - loss: 0.0427 - val_accuracy: 0.9494 - val_loss: 0.1933
Epoch 6/10
243/243 ━━━━━━━━━━━━━━━━━━━━ 18s 75ms/step - accuracy: 0.9870 - loss: 0.0389 - val_accuracy: 0.9231 - val_loss: 0.2334
Epoch 7/10
243/243 ━━━━━━━━━━━━━━━━━━━━ 18s 76ms/step - accuracy: 0.9856 - loss: 0.0409 - val_accuracy: 0.9788 - val_loss: 0.0605
Epoch 8/10
243/243 ━━━━━━━━━━━━━━━━━━━━ 19s 79ms/step - accuracy: 0.9930 - loss: 0.0223 - 

In [8]:
model.save("C:\\Users\\pavani\\Desktop\\python\\voice_gender_classifier.keras")

In [7]:
# gender prediction
from tensorflow.keras.models import load_model

# Load the model once at the start
model_path = r"C:\\Users\\pavani\\Desktop\\python\\voice_gender_classifier.keras"
model = load_model(model_path)

def predict_gender():
    
    audio_path = microphone_audio_input(duration=5)
    # audio_path =r"C:\Users\pavani\Desktop\python\output.wav"
    signal = preprocess_audio(file_path=audio_path)
    # print(signal)
    # print(type(signal))
    # print(signal.shape)
    signal = signal / np.max(np.abs(signal))
    print(signal)
    features = extract_features(signal)

    features = np.expand_dims(features, axis=0)           #(1,features,time frames) - batch added
    features = np.expand_dims(features, axis=-1)          #(1,features,time frames,1) - dim added

    # predicts class probabilities
    prediction = model.predict(features)

    # get predicted class index
    predicted_class = np.argmax(prediction[0])

    # gets prediction to label
    if predicted_class == 0:
        print("Predicted Gender: Male")
    else:
        print("Predicted Gender: Female")

In [13]:
predict_gender()

Recording started for 5 seconds...
Recording finished.
Saved recording to - C:\Users\pavani\Desktop\python\output.wav
[-1.2154832e-06 -3.4869349e-06 -8.9766299e-06 ...  0.0000000e+00
  0.0000000e+00  0.0000000e+00]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
Predicted Gender: Female
